# Phase 3: backtrack and resample when the model is unsure

**Kaggle settings:** GPU T4 x2, Internet ON. No calibration step: the threshold is the one fixed
in Phase 2 (`results/phase2/calibration.json`). Steps: main run on both GPUs, merge, analyze, zip.


In [ ]:
%cd /kaggle/working
!if [ -d seeded-reasoning-lab ]; then cd seeded-reasoning-lab && git pull -q; else git clone -q https://github.com/Mathindata/seeded-reasoning-lab.git; fi
%cd /kaggle/working/seeded-reasoning-lab
!git log --oneline -1
!pip install -q math-verify vllm
import os, sys, json
sys.path.insert(0, "src"); os.environ["PYTHONPATH"] = "src"
BACKEND = "vllm"
print(json.load(open("results/phase2/calibration.json"))["tau"])

In [ ]:
# 1. Main run: one shard per GPU, then merge (resumable: rerun the cell if the session stops)
!(CUDA_VISIBLE_DEVICES=0 python -m sr p3-run --config configs/phase3_s0.json --backend $BACKEND > p3s0.log 2>&1 & \
  CUDA_VISIBLE_DEVICES=1 python -m sr p3-run --config configs/phase3_s1.json --backend $BACKEND > p3s1.log 2>&1 & \
  wait)
!tail -3 p3s0.log p3s1.log
!python -m sr merge --runs runs/phase3_s0 runs/phase3_s1 --out runs/phase3

In [ ]:
# 2. Analysis
from IPython.display import Image, display
from sr.phase3 import analyze3
res = analyze3("runs/phase3")
display(res["summary"].round(3)); display(res["effects"].round(3))
display(res["budget"].round(3)); display(res["curves"].round(3))
display(res["pairing"]); print(res["detection"])
display(Image("runs/phase3/analysis/budget.png"))

In [ ]:
# 3. Save for download (appears in the Output panel on the right)
!cd /kaggle/working/seeded-reasoning-lab && zip -qr /kaggle/working/phase3_runs.zip runs/phase3 p3s0.log p3s1.log
!ls -lh /kaggle/working/phase3_runs.zip